In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Função de circuito IBM

> **Note:** * As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via API do IBM Quantum Platform). Elas estão em status de pré-lançamento e sujeitas a alterações.

## Visão geral
A função de circuito IBM&reg; recebe [PUBs abstratos](./primitive-input-output) como entradas e retorna valores de expectativa mitigados como saídas. Essa função de circuito inclui um pipeline automatizado e personalizado para permitir que pesquisadores se concentrem na descoberta de algoritmos e aplicações.

## Descrição
Após enviar seu PUB, seus circuitos abstratos e observáveis são automaticamente transpilados, executados em hardware e pós-processados para retornar valores de expectativa mitigados. Para isso, combina as seguintes ferramentas:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), incluindo seleção automática de passes de transpilação baseados em IA e heurísticos para traduzir seus circuitos abstratos em circuitos ISA otimizados para hardware
- [Supressão e mitigação de erros necessárias para computação em escala utilitária](./error-mitigation-and-suppression-techniques), incluindo twirling de medições e de portas, desacoplamento dinâmico, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) e Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), para executar PUBs ISA em hardware e retornar valores de expectativa mitigados

![Função de circuito IBM](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Primeiros passos
Autentique-se usando sua [chave de API](http://quantum.cloud.ibm.com/) e selecione a Qiskit Function da seguinte forma. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Exemplo
Para começar, experimente este exemplo básico:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Verifique o [status](/guides/functions#check-job-status) da sua carga de trabalho da Qiskit Function ou obtenha os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Os resultados têm o mesmo formato de um [resultado do Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Entradas
Consulte a tabela a seguir para todos os parâmetros de entrada que a função de circuito IBM aceita. A seção [Opções](#options) seguinte detalha as `options` disponíveis.

| Nome      | Tipo                       | Descrição                                                                                                                                                                                                                         | Obrigatório | Padrão                                                                  | Exemplo                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nome do backend para realizar a consulta.                                                                                                                                                                                              | Sim      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Um iterável de objetos do tipo PUB (primitive unified bloc) abstratos, como tuplas `(circuit, observables)` ou `(circuit, observables, parameter_values)`. Consulte [Visão geral dos PUBs](/guides/primitive-input-output#overview-of-pubs) para mais informações. Os circuitos podem ser abstratos (não-ISA). | Sim      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opções de entrada. Consulte a seção **Opções** para mais detalhes.                                                                                                                                                                                | Não       | Consulte a seção **Opções** para detalhes.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | O nome do recurso de nuvem (cloud resource name) da instância a usar nesse formato.                                                                                                                                                                                        | Não       | Uma instância é escolhida aleatoriamente se sua conta tiver acesso a múltiplas instâncias. | `CRN`                   |

### Opções
#### Estrutura
De forma similar aos primitivos do Qiskit Runtime, as opções da função de circuito IBM podem ser especificadas como um dicionário aninhado. Opções comumente utilizadas, como ``optimization_level`` e ``mitigation_level``, ficam no primeiro nível. Outras opções mais avançadas são agrupadas em diferentes categorias, como ``resilience``.

#### Valores padrão
Se você não especificar um valor para uma opção, o valor padrão definido pelo serviço é utilizado.

#### Nível de mitigação
A função de circuito IBM também suporta `mitigation_level`. O nível de mitigação especifica quanto de supressão e mitigação de erros aplicar ao job. Níveis mais altos geram resultados mais precisos, ao custo de tempos de processamento mais longos. O grau de redução de erros depende do método aplicado. O nível de mitigação abstrai a escolha detalhada dos métodos de mitigação e supressão de erros para permitir que os usuários raciocinem sobre o trade-off custo/precisão apropriado para sua aplicação. A tabela a seguir mostra os métodos correspondentes para cada nível.

> **Note:** Embora os nomes sejam semelhantes, `mitigation_level` aplica técnicas diferentes das usadas pelo `resilience_level` do Estimator.

De forma similar ao ``resilience_level`` no Qiskit Runtime Estimator, o ``mitigation_level`` especifica um conjunto base de opções selecionadas. Quaisquer opções que você especificar manualmente além do nível de mitigação são aplicadas sobre o conjunto base de opções definido pelo nível de mitigação. Portanto, em princípio, você poderia definir o nível de mitigação como 1 e depois desativar a mitigação de medições, embora isso não seja recomendado.

| **Nível de mitigação** | **Técnica** |
|:-:|:-:|
| 1 [Padrão] | Desacoplamento dinâmico + twirling de medições + TREX  |
| 2 | Nível 1 + twirling de portas + ZNE via gate folding |
| 3 | Nível 1 + twirling de portas + ZNE via PEA |

O exemplo a seguir demonstra como definir o nível de mitigação:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Todas as opções disponíveis
Além do `mitigation_level`, a função de circuito IBM também oferece um número selecionado de opções avançadas que permitem ajustar o trade-off custo/precisão. A tabela a seguir mostra todas as opções disponíveis:

| Opção | Sub-opção | Sub-sub-opção | Descrição | Opções disponíveis | Padrão |
|-|-|-|-|-|-|
| default_precision |  |  | A precisão padrão a usar para qualquer PUB ou chamada `run()`<br/>que não especifique uma.<br/>Cada PUB de entrada pode especificar sua própria precisão. Se o método `run()` receber uma precisão, esse valor é usado para todos os PUBs na chamada `run()` que não especificarem a sua.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Tempo máximo de execução em segundos, baseado<br/>no uso do QPU (não no tempo de relógio). O uso do QPU é o<br/>tempo em que o QPU está dedicado a processar seu job. Se um job exceder esse limite de tempo, ele é cancelado forçosamente. | Número inteiro de segundos no intervalo [1, 10800] |  |
| mitigation_level |  |  | Quanto de supressão e mitigação de erros aplicar. Consulte a seção [Nível de mitigação](#mitigation-level) para mais informações sobre os métodos usados em cada nível. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Quanto de otimização realizar nos circuitos. [Níveis mais altos](/guides/set-optimization) geram circuitos mais otimizados, ao custo de maior tempo de transpilação. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Se o desacoplamento dinâmico deve ser ativado. Consulte [Técnicas de supressão e mitigação de erros](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) para a explicação do método.  | True/False | True |
|  | sequence_type |  | Qual sequência de desacoplamento dinâmico usar.<br/>* `XX`: usa a sequência `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: usa a sequência `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: usa a sequência<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Se o twirling de portas Clifford de 2 qubits deve ser aplicado. | True/False | False |
|  | enable_measure |  | Se o twirling de medições deve ser ativado. | True/False | True |
| resilience | measure_mitigation |  | Se o método de mitigação de erros de medição TREX deve ser ativado. Consulte [Técnicas de supressão e mitigação de erros](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) para a explicação do método.  | True/False | True |
|  | zne_mitigation |  | Se o método de mitigação de erros Zero Noise Extrapolation deve ser ativado. Consulte [Técnicas de supressão e mitigação de erros](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) para a explicação do método.  | True/False | False |
|  | zne | amplifier | Qual técnica usar para amplificar o ruído. Uma das seguintes: <br/> - `gate_folding` (padrão) usa gate folding de 2 qubits para amplificar o ruído. Se o fator de ruído exigir amplificação de apenas um subconjunto das portas, essas portas são escolhidas aleatoriamente.<br/><br/> - `gate_folding_front` usa gate folding de 2 qubits para amplificar o ruído. Se o fator de ruído exigir amplificação de apenas um subconjunto das portas, essas portas são selecionadas da frente do circuito DAG ordenado topologicamente.<br/><br/> - `gate_folding_back` usa gate folding de 2 qubits para amplificar o ruído. Se o fator de ruído exigir amplificação de apenas um subconjunto das portas, essas portas são selecionadas do final do circuito DAG ordenado topologicamente.<br/><br/> - `pea` usa uma técnica chamada Probabilistic Error Amplification (PEA) para amplificar o ruído. Consulte [Técnicas de supressão e mitigação de erros](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) para a explicação do método.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Fatores de ruído a usar para amplificação de ruído. | lista de floats; cada float >= 1 | (1, 1.5, 2) para PEA, e (1, 3, 5) nos demais casos. |
|  |  | extrapolator | Fatores de ruído nos quais avaliar os modelos de extrapolação do ajuste. Esta opção não afeta a execução nem o ajuste do modelo de nenhuma forma; ela apenas determina os pontos em que os objetos `extrapolator` são avaliados para serem retornados nos campos de dados chamados `evs_extrapolated` e `stds_extrapolated`. | um ou mais de `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Se o método de mitigação de erros Probabilistic Error Cancellation deve ser ativado. Consulte [Técnicas de supressão e mitigação de erros](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) para a explicação do método.  | True/False | False |
|  | pec | max_overhead | O overhead máximo de amostragem de circuitos permitido, ou `None` para sem máximo. | None/ inteiro >1 | 100 |

No exemplo a seguir, definir o nível de mitigação como 1 desativa inicialmente a mitigação ZNE, mas definir `zne_mitigation` como `True` substitui a configuração relevante do `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Saídas
A saída de uma função de circuito é um [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), que contém dois campos:

- Um ou mais objetos [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Eles podem ser indexados diretamente a partir do `PrimitiveResult`.

- Metadados do nível do job.

Cada `PubResult` contém um campo `data` e um campo `metadata`.

- O campo `data` contém pelo menos um array de valores de expectativa (`PubResult.data.evs`) e um array de erros padrão (`PubResult.data.stds`). Ele também pode conter mais dados, dependendo das opções utilizadas.

- O campo `metadata` contém metadados do nível PUB (`PubResult.metadata`).

O trecho de código a seguir descreve o formato do `PrimitiveResult` (e do `PubResult` associado).